In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c traffic-sign-object-detection-challenge
!unzip traffic-sign-object-detection-challenge.zip

In [ ]:
import json
import os
import shutil
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO


# Configuration
DATASET_DIR = "dataset"
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "train", "images")
TRAIN_ANN_DIR = os.path.join(DATASET_DIR, "train", "annotations.json")
YOLO_DIR = os.path.join(DATASET_DIR, "obj")

# Create YOLO structure
os.makedirs(os.path.join(YOLO_DIR, "images", "train"), exist_ok=True)
os.makedirs(os.path.join(YOLO_DIR, "labels", "train"), exist_ok=True)

# Load COCO annotations
with open(TRAIN_ANN_DIR, "r") as f:
    data = json.load(f)

CLASS_NAMES = {c["id"]: c["name"] for c in data["categories"]}
sorted_ids = sorted(CLASS_NAMES.keys())
class_idx_map = {coco_id: i for i, coco_id in enumerate(sorted_ids)}

# Get image dimensions
img_dict = {img["id"]: img for img in data["images"]}

# Iterate through annotations
for ann in data["annotations"]:
    img_id = ann["image_id"]
    img_info = img_dict[img_id]
    width = img_info["width"]
    height = img_info["height"]
    filename = img_info["file_name"]
    
    # Copy image to YOLO dir
    src_img = os.path.join(TRAIN_IMG_DIR, filename)
    dst_img = os.path.join(YOLO_DIR, "images", "train", filename)
    shutil.copy(src_img, dst_img)
    
    # Convert bbox: COCO is [x, y, w, h] (top-left corner)
    # YOLO is [x_center, y_center, width, height] (normalized)
    bbox = ann["bbox"]
    x, y, w, h = bbox[0], bbox[1], bbox[2], bbox[3]
    
    x_center = (x + w / 2) / width
    y_center = (y + h / 2) / height
    w_norm = w / width
    h_norm = h / height
    
    # Get class index
    class_idx = class_idx_map[ann["category_id"]]
    
    # Write to txt
    txt_path = os.path.join(YOLO_DIR, "labels", "train", filename.replace(".jpg", ".txt"))
    with open(txt_path, "a") as f:
        f.write(f"{class_idx} {x_center} {y_center} {w_norm} {h_norm}\n")

print("✅ Data conversion complete.")


✅ Data conversion complete.


In [2]:
# Configuration
INFER_NAMES = {i: CLASS_NAMES[coco_id] for i, coco_id in enumerate(sorted_ids)}
YOLO_DATA_YAML = f"""
path: dataset/obj
train: images/train
val: images/train
nc: {len(CLASS_NAMES)}
names:
{chr(10).join(f'  {k}: {v}' for k, v in sorted(INFER_NAMES.items()))}
"""

# Save yaml
with open("data.yaml", "w") as f:
    f.write(YOLO_DATA_YAML)

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data="data.yaml",
    epochs=64,
    imgsz=768,
    batch=16,
    name="traffic_signs_v8_n_higher_res",
    patience=8,
    amp=True,
    device=0,
    degrees=5.0,
)
print("Training complete.")

Ultralytics 8.4.38 🚀 Python-3.14.3 torch-2.11.0+rocm7.2 CUDA:0 (AMD Radeon Graphics, 16304MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=64, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=traffic_signs_v8_n_higher_res, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

In [ ]:
# Configuration
TEST_IMG_DIR = "dataset/test/images"
MODEL_PATH = "runs/detect/traffic_signs_v8_n_higher_res/weights/best.pt"
SUBMISSION_CSV = "submission-v8-n-higher-res.csv"

# Load trained model
model = YOLO(MODEL_PATH)
results_list = []

all_files = os.listdir(TEST_IMG_DIR)
print(f"Total files in directory: {len(all_files)}")
print(f"Extensions found: {set(os.path.splitext(f)[1] for f in all_files)}")

test_files = [f for f in all_files if f.lower().endswith(".jpg")]
print(f"JPG files found: {len(test_files)}")

for img_name in tqdm(test_files, desc="Inference"):
    img_path = os.path.join(TEST_IMG_DIR, img_name)
    image_id = os.path.splitext(img_name)[0]  

    try:
        res = model.predict(img_path, verbose=False, iou=0.5)[0]
        boxes = res.boxes

        if boxes is None or len(boxes) == 0:
            results_list.append({
                "image_id": image_id,
                "PredictionString": "No_parking 0.0001 0 0 1 1"
            })
            continue

        preds = []
        for i in range(len(boxes)):
            conf = boxes.conf[i].item()
            cls_id = int(boxes.cls[i].item())
            xyxy = boxes.xyxy[i].cpu().numpy()
            label = INFER_NAMES[cls_id].replace(" ", "_")
            x_min, y_min, x_max, y_max = xyxy
            preds.append(f"{label} {conf:.4f} {x_min:.4f} {y_min:.4f} {x_max:.4f} {y_max:.4f}")

        results_list.append({
            "image_id": image_id,
            "PredictionString": " ".join(preds)
        })

    except Exception as e:
        print(f"⚠️ Failed on {img_name}: {e}")
        results_list.append({
            "image_id": image_id,
            "PredictionString": "No_parking 0.0001 0 0 1 1"
        })

print(f"Rows generated: {len(results_list)}")
assert len(results_list) == 167, f"Expected 167 rows, got {len(results_list)}!"

df = pd.DataFrame(results_list)
df.to_csv(SUBMISSION_CSV, index=False)
print(f"Submission saved to {SUBMISSION_CSV}")

Total files in directory: 167
Extensions found: {'.jpg', '.JPG'}
JPG files found: 167


Inference: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 167/167 [00:10<00:00, 15.70it/s]

Rows generated: 167
✅ Submission saved to submission-v8-n-higher-res.csv


In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
from pathlib import Path

def visualize_predictions(
    results_list: list[dict],
    img_dir: str,
    class_names: dict,          
    num_images: int = 8,
    conf_threshold: float = 0.0,
    cols: int = 4,
    figsize_per_img: tuple = (4, 4),
    seed: int = 42,
):
    """
    Visualize a random sample of test predictions with bounding boxes.

    Args:
        results_list:    Output list from inference loop (dicts with image_id + PredictionString).
        img_dir:         Path to test images directory.
        class_names:     Remapped {int_idx: class_name} dict.
        num_images:      How many images to sample.
        conf_threshold:  Skip boxes below this confidence.
        cols:            Grid columns.
        figsize_per_img: (w, h) in inches per subplot cell.
        seed:            For reproducible random sampling.
    """
    random.seed(seed)

    # Filter to rows that actually have predictions (skip dummy rows)
    valid = [r for r in results_list if not r["PredictionString"].startswith("No_parking 0.0001")]
    sample = random.sample(valid, min(num_images, len(valid)))

    rows = -(-len(sample) // cols)  # ceiling division
    fig, axes = plt.subplots(
        rows, cols,
        figsize=(figsize_per_img[0] * cols, figsize_per_img[1] * rows)
    )
    axes = axes.flatten() if rows * cols > 1 else [axes]

    # Assign a consistent color per class
    unique_classes = list(set(class_names.values()))
    palette = {cls: (random.random(), random.random(), random.random()) for cls in unique_classes}

    for ax, record in zip(axes, sample):
        img_path = Path(img_dir) / f"{record['image_id']}.jpg"
        if not img_path.exists():
            ax.axis("off")
            continue

        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h_img, w_img = img_rgb.shape[:2]

        ax.imshow(img_rgb)
        ax.set_title(record["image_id"], fontsize=7, pad=2)
        ax.axis("off")

        pred_str = record["PredictionString"].strip()
        if not pred_str:
            continue

        tokens = pred_str.split()
        # Each prediction: label conf x_min y_min x_max y_max  → 6 tokens
        import re
        predictions = re.findall(
            r'([A-Za-z][A-Za-z0-9_ ]*?)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)',
            pred_str
        )
        for label, conf, x_min, y_min, x_max, y_max in predictions:
            label = label.strip()
            conf  = float(conf)
            x_min, y_min, x_max, y_max = float(x_min), float(y_min), float(x_max), float(y_max)

            if conf < conf_threshold:
                continue

            color = palette.get(label, (1, 0, 0))
            rect = patches.Rectangle(
                (x_min, y_min), x_max - x_min, y_max - y_min,
                linewidth=1.5, edgecolor=color, facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(
                x_min, max(y_min - 4, 0),
                f"{label} {conf:.2f}",
                fontsize=6, color="white",
                bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor="none")
            )

    # Hide unused subplot slots
    for ax in axes[len(sample):]:
        ax.axis("off")

    plt.suptitle("Test Set Predictions", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig("predictions_preview.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved → predictions_preview.png")

visualize_predictions(
    results_list=results_list,
    img_dir=TEST_IMG_DIR,
    class_names=INFER_NAMES, 
    num_images=8,
    conf_threshold=0.25,
    cols=4,
)


<Figure size 1600x800 with 8 Axes>

✅ Saved → predictions_preview.png
